load data

In [4]:
pip install transformers datasets pandas torch scikit-learn

  Using cached transformers-5.11.0-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.7.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py

In [6]:
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm

# 1. Load your data (assuming it's named 'goemotions_data.csv')
df = pd.read_csv(r"C:\Users\kasih\nlp_project\data\goemotions_1.csv")

# 2. Define the exact list of 28 emotion columns present in your dataset
emotion_columns = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

# 3. Initialize a Hugging Face pipeline that matches your dataset's emotions
# bhadresh-savani/bert-base-go-emotion outputs predictions for these exact classes
classifier = pipeline(
    "text-classification", 
    model="bhadresh-savani/bert-base-go-emotion", 
    top_k=None, # Returns probabilities for ALL 28 emotions
    device=-1   # Set to 0 if you are using a GPU
)

print("Sample Text:", df['text'].iloc[0])
print("True Emotion:", [col for col in emotion_columns if df[col].iloc[0] == 1])

c:\Users\kasih\nlp_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kasih\.cache\huggingface\hub\models--bhadresh-savani--bert-base-go-emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 43526.00it/s]


Sample Text: That game hurt.
True Emotion: ['sadness']


batch preprocessing

In [ ]:
# Extract texts as a list
texts = df["text"].astype(str).tolist()

# Store predictions matrix (initially all zeros)
predicted_matrix = np.zeros((len(df), len(emotion_columns)))

print("Running deep learning inference over dataset...")
# Process in batches for speed
batch_size = 32

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    batch_preds = classifier(batch_texts)
    
    for batch_idx, pred_list in enumerate(batch_preds):
        # pred_list looks like: [{'label': 'sadness', 'score': 0.92}, {'label': 'anger', 'score': 0.05}, ...]
        row_idx = i + batch_idx
        
        for pred in pred_list:
            emotion_name = pred['label']
            confidence = pred['score']
            
            # If the model is confident (> 0.3), mark it as 1
            # Multi-label thresholds are usually lower than 0.5 because probabilities split across 28 classes
            if confidence > 0.3 and emotion_name in emotion_columns:
                col_idx = emotion_columns.index(emotion_name)
                predicted_matrix[row_idx, col_idx] = 1

# Create a dataframe of predictions
df_preds = pd.DataFrame(predicted_matrix, columns=[f"pred_{col}" for col in emotion_columns])

# Combine original data with predictions
df_final = pd.concat([df, df_preds], axis=1)
df_final.to_csv("goemotions_analyzed.csv", index=False)
print("Analysis saved to goemotions_analyzed.csv")

Running deep learning inference over dataset...


  2%|▏         | 48/2188 [03:02<2:17:43,  3.86s/it]

evaluation

In [ ]:
from sklearn.metrics import classification_report

# Extract true matrix and predicted matrix
y_true = df[emotion_columns].values
y_pred = df_final[[f"pred_{col}" for col in emotion_columns]].values

# Generate report
print("\n===== ADVANCED DEEP LEARNING MODEL PERFORMANCE =====")
print(classification_report(y_true, y_pred, target_names=emotion_columns, zero_division=0))

save model

In [ ]:
save_directory = "../models/my_saved_emotion_model"
classifier.save_pretrained(save_directory)

print(f"Model and tokenizer successfully saved to {save_directory}!")